In [1]:
import json
import sys

from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.rgrag_retriever import RGRAGRetriever
from src.rgrag_edu_retriever import RGRAGEDURetriever

d:\Dev\tcc\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Dataset

In [2]:
dataset_path = '../artifacts/dataset/MultiHopRAG.json'
with open(dataset_path, 'r', encoding='utf-8') as file:
    dataset = json.load(file)

In [3]:
valid_dataset = [
    item
    for item in dataset
    if item['question_type'] != 'null_query'
    and item['evidence_list']
]

In [4]:
print('Original dataset size:', len(dataset))
print('Valid dataset size:', len(valid_dataset))

Original dataset size: 2556
Valid dataset size: 2255


## Load Retriever

In [5]:
retriever = RGRAGRetriever(
    workspace_path=project_root / 'graphrag_workspace',
    rst_edges_path=project_root / 'artifacts' / 'rgrag' / 'rst_relationship_edges.jsonl',
    consolidated_path=project_root / 'artifacts' / 'rgrag' / 'relationship_edu_consolidated.jsonl',
    community_level=2,
)

edu_retriever = RGRAGEDURetriever(
    workspace_path=project_root / 'graphrag_workspace',
    rst_edges_path=project_root / 'artifacts' / 'rgrag' / 'rst_relationship_edges.jsonl',
    consolidated_path=project_root / 'artifacts' / 'rgrag' / 'relationship_edu_consolidated.jsonl',
    rst_path=project_root / 'artifacts' / 'rst' / 'multihop_rst.jsonl',
    embeddings_path=project_root / 'artifacts' / 'rgrag' / 'edu_embeddings.npy',
    index_path=project_root / 'artifacts' / 'rgrag' / 'edu_index.json',
    community_level=2,
)

In [6]:
def normalize_text(text: str) -> str:
    return text.replace(' ', '').replace('\n', '')

def get_texts(text_unit_ids, retriever):
    texts = []

    for text_unit_id in text_unit_ids:
        text_unit = retriever.ranker.text_units.get(str(text_unit_id))

        if text_unit is not None:
            texts.append(text_unit.text)

    return texts

def find_gold_facts(evidence_list, texts):
    normalized_texts = [normalize_text(text) for text in texts]

    results = []

    for evidence in evidence_list:
        fact = evidence['fact']
        normalized_fact = normalize_text(fact)

        found = any(
            normalized_fact in text
            for text in normalized_texts
        )

        results.append({
            'fact': fact,
            'title': evidence.get('title'),
            'source': evidence.get('source'),
            'found': found,
        })

    return results

def evaluate_candidate_recall(item):
    query = item['query']
    evidence_list = item['evidence_list']

    result = retriever.retrieve_candidates(query)

    baseline_ids = set(result['graphrag']['text_unit_ids'])
    rst_ids = set(result['rst']['text_unit_ids'])
    rgrag_ids = baseline_ids | rst_ids

    baseline_texts = get_texts(baseline_ids, retriever)
    rgrag_texts = get_texts(rgrag_ids, retriever)

    baseline_gold = find_gold_facts(evidence_list, baseline_texts)
    rgrag_gold = find_gold_facts(evidence_list, rgrag_texts)

    total_gold = len(evidence_list)
    baseline_found = sum(item['found'] for item in baseline_gold)
    rgrag_found = sum(item['found'] for item in rgrag_gold)

    return {
        'query': query,
        'question_type': item['question_type'],
        'gold_count': total_gold,
        'graphrag_candidates': len(baseline_ids),
        'rst_candidates': len(rst_ids),
        'rst_new_candidates': len(rst_ids - baseline_ids),
        'rgrag_candidates': len(rgrag_ids),
        'graphrag_found': baseline_found,
        'rgrag_found': rgrag_found,
        'graphrag_recall': baseline_found / total_gold,
        'rgrag_recall': rgrag_found / total_gold,
        'graphrag_complete': baseline_found == total_gold,
        'rgrag_complete': rgrag_found == total_gold,
        'new_gold_found': rgrag_found - baseline_found,
        'graphrag_gold': baseline_gold,
        'rgrag_gold': rgrag_gold,
    }

def find_text_units_with_fact(fact, text_unit_ids):
    normalized_fact = normalize_text(fact)
    matches = []

    for text_unit_id in text_unit_ids:
        text_unit = retriever.ranker.text_units.get(str(text_unit_id))

        if text_unit is None:
            continue

        if normalized_fact in normalize_text(text_unit.text):
            matches.append({
                'text_unit_id': str(text_unit_id),
                'text': text_unit.text,
            })

    return matches

def evaluate_ranking(item, top_k=10):
    result = retriever.retrieve(
        query=item['query'],
        top_k=top_k,
    )

    gold = [
        evidence['fact']
        for evidence in item['evidence_list']
    ]

    return {
        'query': item['query'],
        'question_type': item['question_type'],
        'gold': gold,
        'graphrag': result['baseline'],
        'rgrag': result['rgrag'],
    }

def calculate_metrics(retrieved_lists, gold_lists):
    hits_at_10_count = 0
    hits_at_4_count = 0
    map_at_10_list = []
    mrr_list = []

    for retrieved, gold in zip(retrieved_lists, gold_lists):
        hits_at_10_flag = False
        hits_at_4_flag = False
        average_precision_sum = 0
        first_relevant_rank = None
        find_gold = []

        gold = [normalize_text(item) for item in gold]
        retrieved = [normalize_text(item) for item in retrieved]

        for rank, retrieved_item in enumerate(retrieved[:10], start=1):
            if any(gold_item in retrieved_item for gold_item in gold):
                hits_at_10_flag = True

                if first_relevant_rank is None:
                    first_relevant_rank = rank

                if rank <= 4:
                    hits_at_4_flag = True

                count = 0

                for gold_item in gold:
                    if gold_item in retrieved_item and gold_item not in find_gold:
                        count += 1
                        find_gold.append(gold_item)

                average_precision_sum += count / rank

        hits_at_10_count += int(hits_at_10_flag)
        hits_at_4_count += int(hits_at_4_flag)
        map_at_10_list.append(average_precision_sum / min(len(gold), 10))
        mrr_list.append(1 / first_relevant_rank if first_relevant_rank else 0)

    return {
        'Hits@4': hits_at_4_count / len(gold_lists),
        'Hits@10': hits_at_10_count / len(gold_lists),
        'MAP@10': sum(map_at_10_list) / len(gold_lists),
        'MRR@10': sum(mrr_list) / len(gold_lists),
    }

def complete_at_k(retrieved_lists, gold_lists, k):
    complete = 0

    for retrieved, gold in zip(retrieved_lists, gold_lists):
        retrieved = [
            normalize_text(text)
            for text in retrieved[:k]
        ]

        gold = [
            normalize_text(fact)
            for fact in gold
        ]

        found = {
            gold_item
            for gold_item in gold
            if any(
                gold_item in retrieved_item
                for retrieved_item in retrieved
            )
        }

        if len(found) == len(gold):
            complete += 1

    return complete / len(gold_lists)

def get_gold_ranks(item):
    candidates = retriever.retrieve_candidates(item['query'])

    baseline_ids = set(candidates['graphrag']['text_unit_ids'])
    rst_ids = set(candidates['rst']['text_unit_ids'])
    rgrag_ids = baseline_ids | rst_ids

    baseline_ranked = retriever.ranker.rank(
        query=item['query'],
        text_unit_ids=baseline_ids,
        baseline_ids=baseline_ids,
        rst_ids=set(),
        top_k=len(baseline_ids),
    )

    rgrag_ranked = retriever.ranker.rank(
        query=item['query'],
        text_unit_ids=rgrag_ids,
        baseline_ids=baseline_ids,
        rst_ids=rst_ids,
        top_k=len(rgrag_ids),
    )

    rows = []

    for evidence in item['evidence_list']:
        fact = normalize_text(evidence['fact'])

        baseline_matches = [
            (rank, result)
            for rank, result in enumerate(baseline_ranked, start=1)
            if fact in normalize_text(result['text'])
        ]

        rgrag_matches = [
            (rank, result)
            for rank, result in enumerate(rgrag_ranked, start=1)
            if fact in normalize_text(result['text'])
        ]

        rows.append({
            'source': evidence['source'],
            'title': evidence['title'],
            'graphrag_rank': baseline_matches[0][0] if baseline_matches else None,
            'rgrag_rank': rgrag_matches[0][0] if rgrag_matches else None,
            'rgrag_origin': rgrag_matches[0][1]['origin'] if rgrag_matches else None,
            'rgrag_score': rgrag_matches[0][1]['score'] if rgrag_matches else None,
            'fact': evidence['fact'],
        })

    return pd.DataFrame(rows)

## RGRAG Structure

In [17]:
results = []

for item in tqdm(valid_dataset[:20]):
    results.append(
        evaluate_candidate_recall(item)
    )

100%|██████████| 20/20 [00:05<00:00,  3.77it/s]


In [18]:
df = pd.DataFrame([
    {
        key: value
        for key, value in result.items()
        if key not in {'graphrag_gold', 'rgrag_gold'}
    }
    for result in results
])

df

,query,question_type,gold_count,graphrag_candidates,rst_candidates,rst_new_candidates,rgrag_candidates,graphrag_found,rgrag_found,graphrag_recall,rgrag_recall,graphrag_complete,rgrag_complete,new_gold_found
0,Who is the individual associated with the cryp...,inference_query,3,42,49,11,53,3,3,1.000000,1.000000,True,True,0
1,Which individual is implicated in both inflati...,inference_query,2,40,48,28,68,2,2,1.000000,1.000000,True,True,0
2,Who is the figure associated with generative A...,inference_query,2,35,42,11,46,2,2,1.000000,1.000000,True,True,0
3,Do the TechCrunch article on software companie...,comparison_query,2,53,65,33,86,2,2,1.000000,1.000000,True,True,0
4,Which online betting platform provides a welco...,inference_query,3,39,36,15,54,3,3,1.000000,1.000000,True,True,0
5,Who is the individual alleged to have built a ...,inference_query,2,42,58,21,63,2,2,1.000000,1.000000,True,True,0
6,Does the TechCrunch article on Twitch's subscr...,comparison_query,2,80,99,44,124,2,2,1.000000,1.000000,True,True,0
7,Does 'The New York Times' article attribute th...,comparison_query,2,134,151,31,165,2,2,1.000000,1.000000,True,True,0
8,What is the name of the organization discussed...,inference_query,4,63,60,36,99,3,4,0.750000,1.000000,False,True,1
9,"Which company, as reported by both TechCrunch ...",inference_query,3,207,164,48,255,3,3,1.000000,1.000000,True,True,0


In [19]:
summary = pd.Series({
    'queries': len(df),
    'graphrag_candidate_recall': df['graphrag_recall'].mean(),
    'rgrag_candidate_recall': df['rgrag_recall'].mean(),
    'graphrag_complete_recall': df['graphrag_complete'].mean(),
    'rgrag_complete_recall': df['rgrag_complete'].mean(),
    'queries_improved': (df['rgrag_recall'] > df['graphrag_recall']).sum(),
    'new_gold_facts_found': df['new_gold_found'].sum(),
    'avg_graphrag_candidates': df['graphrag_candidates'].mean(),
    'avg_rgrag_candidates': df['rgrag_candidates'].mean(),
    'avg_rst_new_candidates': df['rst_new_candidates'].mean(),
})

summary

queries                       20.000000
graphrag_candidate_recall      0.841667
rgrag_candidate_recall         0.879167
graphrag_complete_recall       0.700000
rgrag_complete_recall          0.800000
queries_improved               2.000000
new_gold_facts_found           2.000000
avg_graphrag_candidates       81.500000
avg_rgrag_candidates         121.400000
avg_rst_new_candidates        39.900000
dtype: float64

In [20]:
improved = df[
    df['rgrag_recall'] > df['graphrag_recall']
]

improved[
    [
        'question_type',
        'gold_count',
        'graphrag_found',
        'rgrag_found',
        'new_gold_found',
        'graphrag_candidates',
        'rgrag_candidates',
        'query',
    ]
]

,question_type,gold_count,graphrag_found,rgrag_found,new_gold_found,graphrag_candidates,rgrag_candidates,query
8,inference_query,4,3,4,1,63,99,What is the name of the organization discussed...
10,comparison_query,2,1,2,1,55,92,Does 'The Age' article suggest that Australia'...


In [21]:
for index in [8, 10]:
    result = results[index]

    print()
    print('QUERY', index)
    print(result['query'])

    for graphrag_gold, rgrag_gold in zip(
        result['graphrag_gold'],
        result['rgrag_gold'],
    ):
        if not graphrag_gold['found'] and rgrag_gold['found']:
            print()
            print('NEW GOLD')
            print('Source:', rgrag_gold['source'])
            print('Title:', rgrag_gold['title'])
            print('Fact:', rgrag_gold['fact'])


QUERY 8
What is the name of the organization discussed in TechCrunch articles that, despite its financial instability, is recognized for creating ChatGPT, which is both a priority and a platform for ongoing innovations, and is planning to enhance its capabilities with the release of GPT-4 and associated APIs?

NEW GOLD
Source: TechCrunch
Title: WTF is going on at OpenAI? We have theories
Fact: Despite being the hottest tech company in the world right now and everyone talking about ChatGPT, OpenAI isn’t exactly a sound business.

QUERY 10
Does 'The Age' article suggest that Australia's Davis Cup team is aiming for an improvement in their performance compared to the previous year, while the 'Sporting News' article indicates that the South Africa national rugby team has already achieved an improvement to reach the Rugby World Cup semi-finals?

NEW GOLD
Source: The Age
Title: ‘Biggest win of my career’: De Minaur, Popyrin power Australia into Davis Cup final
Fact: “Hopefully we can go one

In [23]:
for index in [8, 10]:
    item = valid_dataset[index]
    candidate_result = retriever.retrieve_candidates(item['query'])

    baseline_ids = set(candidate_result['graphrag']['text_unit_ids'])
    rst_only_ids = set(candidate_result['rst']['text_unit_ids']) - baseline_ids

    evaluation = results[index]

    for graphrag_gold, rgrag_gold in zip(
        evaluation['graphrag_gold'],
        evaluation['rgrag_gold'],
    ):
        if not graphrag_gold['found'] and rgrag_gold['found']:
            matches = find_text_units_with_fact(
                rgrag_gold['fact'],
                rst_only_ids,
            )

            print()
            print('QUERY', index)
            print('Fact:', rgrag_gold['fact'])

            for match in matches:
                print('TextUnit:', match['text_unit_id'])
                print(match['text'][:500])


QUERY 8
Fact: Despite being the hottest tech company in the world right now and everyone talking about ChatGPT, OpenAI isn’t exactly a sound business.
TextUnit: d401a3ec00e1676b9501da84b2458767798e4c4b247804b1732632c8a7a2099cfa80ea6654293ba8a5d4add2f39f1c60857204238b9a10a674c89f483dc1bd53
In perhaps the most unexpected tech news of the year, billionaire and AI evangelist Sam Altman has been ejected from his CEO role at OpenAI by the company’s board after an apparent vote of no confidence. Its exact wording in a release issued this afternoon: Altman’s “departure follows a deliberative review process by the board, which concluded that he was not consistently candid in his communications with the board, hindering its ability to exercise its responsibilities.”

What the hell is happe

QUERY 10
Fact: “Hopefully we can go one better this year,” he added, recalling the 2-0 defeat by Canada in 2022.
TextUnit: 3f26a80ba2f66a4c66bd56abb864a2eb815abe7e884c8b3e3ee59e6d19febf4d78128c0017ddd37b0f15

In [25]:
for index in [8, 10]:
    ranking = evaluate_ranking(valid_dataset[index])

    print()
    print('QUERY', index)
    print(ranking['query'])

    print()
    print('GRAPHRAG')

    for rank, item in enumerate(ranking['graphrag'], start=1):
        found = any(
            normalize_text(fact) in normalize_text(item['text'])
            for fact in ranking['gold']
        )

        if found:
            print(rank, round(item['score'], 4), 'GOLD')

    print()
    print('RGRAG')

    for rank, item in enumerate(ranking['rgrag'], start=1):
        found = any(
            normalize_text(fact) in normalize_text(item['text'])
            for fact in ranking['gold']
        )

        if found:
            print(rank, item['origin'], round(item['score'], 4), 'GOLD')


QUERY 8
What is the name of the organization discussed in TechCrunch articles that, despite its financial instability, is recognized for creating ChatGPT, which is both a priority and a platform for ongoing innovations, and is planning to enhance its capabilities with the release of GPT-4 and associated APIs?

GRAPHRAG
3 0.6467 GOLD
8 0.6275 GOLD

RGRAG
3 both 0.6467 GOLD
10 both 0.6275 GOLD

QUERY 10
Does 'The Age' article suggest that Australia's Davis Cup team is aiming for an improvement in their performance compared to the previous year, while the 'Sporting News' article indicates that the South Africa national rugby team has already achieved an improvement to reach the Rugby World Cup semi-finals?

GRAPHRAG

RGRAG


In [ ]:
ranking_results = []

for item in tqdm(valid_dataset[:20]):
    ranking_results.append(
        evaluate_ranking(item, top_k=10)
    )

In [ ]:
gold_lists = [
    result['gold']
    for result in ranking_results
]

graphrag_lists = [
    [
        item['text']
        for item in result['graphrag']
    ]
    for result in ranking_results
]

rgrag_lists = [
    [
        item['text']
        for item in result['rgrag']
    ]
    for result in ranking_results
]

In [ ]:
graphrag_metrics = calculate_metrics(
    graphrag_lists,
    gold_lists,
)

rgrag_metrics = calculate_metrics(
    rgrag_lists,
    gold_lists,
)

pd.DataFrame({
    'GraphRAG': graphrag_metrics,
    'RGRAG': rgrag_metrics,
})

In [30]:
get_gold_ranks(valid_dataset[8])

,source,title,graphrag_rank,rgrag_rank,rgrag_origin,rgrag_score,fact
0,TechCrunch,WTF is going on at OpenAI? We have theories,NaN,37,rst,0.507493,Despite being the hottest tech company in the ...
1,TechCrunch,How the OpenAI fiasco could bolster Meta and t...,16.0,22,both,0.536512,"It has been a whirlwind four days for OpenAI, ..."
2,TechCrunch,"One year later, ChatGPT is still alive and kic...",3.0,3,both,0.646719,"Indeed, ChatGPT became priority number one at ..."
3,TechCrunch,ChatGPT: Everything you need to know about the...,8.0,10,both,0.627532,OpenAI announced that GPT-4 with vision will b...


In [31]:
get_gold_ranks(valid_dataset[10])

,source,title,graphrag_rank,rgrag_rank,rgrag_origin,rgrag_score,fact
0,The Age,"‘Biggest win of my career’: De Minaur, Popyrin...",NaN,21,rst,0.457039,"“Hopefully we can go one better this year,” he..."
1,Sporting News,Where to watch England vs South Africa: Live s...,12.0,13,both,0.484997,England will no doubt be looking to echo the s...


In [32]:
ranking_results = []

for item in tqdm(valid_dataset[:20]):
    ranking_results.append(
        evaluate_ranking(item, top_k=10)
    )

gold_lists = [
    result['gold']
    for result in ranking_results
]

graphrag_lists = [
    [item['text'] for item in result['graphrag']]
    for result in ranking_results
]

rgrag_lists = [
    [item['text'] for item in result['rgrag']]
    for result in ranking_results
]

graphrag_metrics = calculate_metrics(
    graphrag_lists,
    gold_lists,
)

rgrag_metrics = calculate_metrics(
    rgrag_lists,
    gold_lists,
)

pd.DataFrame({
    'GraphRAG': graphrag_metrics,
    'RGRAG': rgrag_metrics,
})

100%|██████████| 20/20 [00:04<00:00,  4.75it/s]


,GraphRAG,RGRAG
Hits@4,0.750000,0.750000
Hits@10,0.900000,0.900000
MAP@10,0.350863,0.345938
MRR@10,0.649167,0.640833


In [33]:
pd.Series({
    'GraphRAG Complete@4': complete_at_k(
        graphrag_lists,
        gold_lists,
        4,
    ),
    'RGRAG Complete@4': complete_at_k(
        rgrag_lists,
        gold_lists,
        4,
    ),
    'GraphRAG Complete@10': complete_at_k(
        graphrag_lists,
        gold_lists,
        10,
    ),
    'RGRAG Complete@10': complete_at_k(
        rgrag_lists,
        gold_lists,
        10,
    ),
})

GraphRAG Complete@4     0.20
RGRAG Complete@4        0.20
GraphRAG Complete@10    0.35
RGRAG Complete@10       0.35
dtype: float64

## RGRAG EDU Aware

In [13]:
item = valid_dataset[10]

result = edu_retriever.retrieve(
    item['query'],
    top_k=10,
)

print('GraphRAG candidates:', result['graphrag_candidates'])
print('RST new candidates:', result['rst_new_candidates'])
print('RGRAG candidates:', result['rgrag_candidates'])
print('EDU covered candidates:', result['edu_covered_candidates'])

print()

for rank, retrieved in enumerate(result['ranked'], start=1):
    gold = any(
        normalize_text(evidence['fact'])
        in normalize_text(retrieved['text'])
        for evidence in item['evidence_list']
    )

    print(
        rank,
        'GOLD' if gold else '',
        retrieved['origin'],
        retrieved['score_source'],
        round(retrieved['score'], 4),
        '| TU:',
        round(retrieved['text_unit_score'], 4),
        '| EDU:',
        round(retrieved['edu_score'], 4)
        if retrieved['edu_score'] is not None
        else None,
    )

    if retrieved['score_source'] == 'edu':
        print('Best EDU:', retrieved['best_edu_text'])

GraphRAG candidates: 55
RST new candidates: 37
RGRAG candidates: 92
EDU covered candidates: 92

1  both edu 0.5943 | TU: 0.5227 | EDU: 0.5943
Best EDU: South Africa have already shown they can match, and beat, the world's best,
2 GOLD rst edu 0.5905 | TU: 0.457 | EDU: 0.5905
Best EDU: as they seek to lift the Davis Cup for the first time in 20 years.
3  both edu 0.5849 | TU: 0.5237 | EDU: 0.5849
Best EDU: Those successes played out in the recent Rugby World Cup,
4  both edu 0.5744 | TU: 0.463 | EDU: 0.5744
Best EDU: as they face South Africa in the semi-final of the Rugby World Cup and in the ODI Cricket World Cup.
5  both edu 0.5581 | TU: 0.5017 | EDU: 0.5581
Best EDU: everyone is expecting a New Zealand and South Africa final.
6  both edu 0.5576 | TU: 0.4098 | EDU: 0.5576
Best EDU: they are in reasonable shape ahead of nexr year’s World Cup.
7  both edu 0.5466 | TU: 0.4853 | EDU: 0.5466
Best EDU: to see Australia fulfil its footballing potential
8  both text_unit 0.5446 | TU: 0.5446 

In [14]:
result = edu_retriever.retrieve(
    item['query'],
    top_k=1000,
)

ranked = result['ranked']

for evidence in item['evidence_list']:
    fact = normalize_text(evidence['fact'])

    matches = [
        (rank, retrieved)
        for rank, retrieved in enumerate(ranked, start=1)
        if fact in normalize_text(retrieved['text'])
    ]

    print()
    print('Source:', evidence['source'])
    print('Title:', evidence['title'])

    if not matches:
        print('Rank: NOT FOUND')
        continue

    rank, retrieved = matches[0]

    print('Rank:', rank)
    print('Origin:', retrieved['origin'])
    print('Score source:', retrieved['score_source'])
    print('Final score:', round(retrieved['score'], 4))
    print('TU score:', round(retrieved['text_unit_score'], 4))
    print(
        'EDU score:',
        round(retrieved['edu_score'], 4)
        if retrieved['edu_score'] is not None
        else None,
    )

    if retrieved['best_edu_text']:
        print('Best EDU:', retrieved['best_edu_text'])


Source: The Age
Title: ‘Biggest win of my career’: De Minaur, Popyrin power Australia into Davis Cup final
Rank: 2
Origin: rst
Score source: edu
Final score: 0.5905
TU score: 0.457
EDU score: 0.5905
Best EDU: as they seek to lift the Davis Cup for the first time in 20 years.

Source: Sporting News
Title: Where to watch England vs South Africa: Live stream, TV channel, lineups, odds for 2023 Rugby World Cup semifinal
Rank: 10
Origin: both
Score source: edu
Final score: 0.5434
TU score: 0.485
EDU score: 0.5434
Best EDU: changing 3️⃣ of his England starters for Saturday's Rugby World Cup semi-final,


In [11]:
item = valid_dataset[8]

candidates = edu_retriever.rgrag.retrieve_candidates(item['query'])

graphrag_result = candidates['graphrag']
rst_result = candidates['rst']

text_unit_edu_rows = edu_retriever.ranker.build_text_unit_edu_rows(
    graphrag_result=graphrag_result,
    rst_result=rst_result,
)

fact = item['evidence_list'][0]['fact']
fact_normalized = normalize_text(fact)

rgrag_ids = set(graphrag_result['text_unit_ids']) | set(rst_result['text_unit_ids'])

gold_text_unit_id = next(
    text_unit_id
    for text_unit_id in rgrag_ids
    if fact_normalized in normalize_text(
        edu_retriever.rgrag.ranker.text_units[str(text_unit_id)].text
    )
)

print('Gold TextUnit:', gold_text_unit_id)
print()

rows = text_unit_edu_rows.get(str(gold_text_unit_id), set())

for row in rows:
    edu_key = edu_retriever.ranker.row_to_edu_key[row]
    edu_text = edu_retriever.ranker.edu_texts[edu_key]

    print(edu_key)
    print(edu_text)
    print()

TypeError: EDUTextUnitRanker.build_text_unit_edu_rows() got an unexpected keyword argument 'graphrag_result'

In [52]:
gold_text_unit = edu_retriever.rgrag.ranker.text_units[gold_text_unit_id]

print(type(gold_text_unit))
print()
print(gold_text_unit)
print()
print(gold_text_unit.__dict__)

<class 'graphrag.data_model.text_unit.TextUnit'>

TextUnit(id='d401a3ec00e1676b9501da84b2458767798e4c4b247804b1732632c8a7a2099cfa80ea6654293ba8a5d4add2f39f1c60857204238b9a10a674c89f483dc1bd53', short_id='886', text='In perhaps the most unexpected tech news of the year, billionaire and AI evangelist Sam Altman has been ejected from his CEO role at OpenAI by the company’s board after an apparent vote of no confidence. Its exact wording in a release issued this afternoon: Altman’s “departure follows a deliberative review process by the board, which concluded that he was not consistently candid in his communications with the board, hindering its ability to exercise its responsibilities.”\n\nWhat the hell is happening at the most hyped company in the world?! Here are some totally speculative theories that occurred to us and others around the web.\n\n1. Did Altman circumvent the board in a major deal?\n\nBased on the board’s language and the way these giant tech companies work, this is the p

In [53]:
import pandas as pd

documents_path = project_root / 'graphrag_workspace' / 'output' / 'documents.parquet'

documents_df = pd.read_parquet(documents_path)

print(documents_df.columns.tolist())
print()
print(documents_df.iloc[0])

['id', 'human_readable_id', 'title', 'text', 'text_unit_ids', 'creation_date', 'raw_data']

id                   6cd0f2d7fa474fdb6fd71ad3b35bb9eb79bf1ab5aa7511...
human_readable_id                                                 None
title                200+ of the best deals from Amazon's Cyber Mon...
text                 Table of Contents Table of Contents Echo, Fire...
text_unit_ids        [31aea63d95e9892b2589cecbc8a68507943b78845e628...
creation_date                                2023-11-27T08:45:59+00:00
raw_data             {'title': '200+ of the best deals from Amazon'...
Name: 0, dtype: object


In [54]:
document_id = 'fc5bab5247d6fb652f13204cda1b12a56ac172989a9ec284ceff1078aa9c2974'

document_row = documents_df[
    documents_df['id'].astype(str) == document_id
]

print(document_row)

                                                    id human_readable_id  \
333  fc5bab5247d6fb652f13204cda1b12a56ac172989a9ec2...              None   

                                           title  \
333  WTF is going on at OpenAI? We have theories   

                                                  text  \
333  In perhaps the most unexpected tech news of th...   

                                         text_unit_ids  \
333  [d401a3ec00e1676b9501da84b2458767798e4c4b24780...   

                 creation_date  \
333  2023-11-18T00:09:53+00:00   

                                              raw_data  
333  {'title': 'WTF is going on at OpenAI? We have ...  


In [55]:
document_id = 'fc5bab5247d6fb652f13204cda1b12a56ac172989a9ec284ceff1078aa9c2974'

document = documents_df[
    documents_df['id'].astype(str) == document_id
].iloc[0]

document_text = document['text']

gold_text_unit = edu_retriever.rgrag.ranker.text_units[
    'd401a3ec00e1676b9501da84b2458767798e4c4b247804b1732632c8a7a2099cfa80ea6654293ba8a5d4add2f39f1c60857204238b9a10a674c89f483dc1bd53'
]

start = document_text.find(gold_text_unit.text)

print('Document length:', len(document_text))
print('TextUnit length:', len(gold_text_unit.text))
print('TextUnit start:', start)
print('TextUnit end:', start + len(gold_text_unit.text) if start >= 0 else None)

Document length: 8895
TextUnit length: 5798
TextUnit start: 0
TextUnit end: 5798


In [56]:
import json

rst_path = project_root / 'artifacts' / 'rst' / 'multihop_rst.jsonl'

rst_document = None

with open(rst_path, 'r', encoding='utf-8') as file:
    for line in file:
        line = line.strip()

        if not line:
            continue

        item = json.loads(line)

        if str(item['document_id']) == document_id:
            rst_document = item
            break

tu_start = start
tu_end = start + len(gold_text_unit.text)

tu_edus = [
    edu
    for edu in rst_document['edus']
    if edu['start'] < tu_end and edu['end'] > tu_start
]

print('EDUs inside TextUnit:', len(tu_edus))
print()

for edu in tu_edus:
    if 'sound business' in edu['text']:
        print('GOLD EDU FOUND')
        print('EDU ID:', edu['edu_id'])
        print('Start:', edu['start'])
        print('End:', edu['end'])
        print('Text:', edu['text'])

EDUs inside TextUnit: 118

GOLD EDU FOUND
EDU ID: 62
Start: 2668
End: 2706
Text: OpenAI isn’t exactly a sound business.


In [57]:
total_edus = 0
unique_edus = set()

with open(rst_path, 'r', encoding='utf-8') as file:
    for line in file:
        line = line.strip()

        if not line:
            continue

        item = json.loads(line)

        for edu in item['edus']:
            total_edus += 1
            unique_edus.add(edu['edu_key'])

print('Total EDUs:', total_edus)
print('Unique EDUs:', len(unique_edus))

Total EDUs: 160839
Unique EDUs: 160839


In [58]:
gold_edu_key = f'{document_id}:62'

print('Gold EDU in current embedding index:', gold_edu_key in edu_retriever.ranker.edu_index)

Gold EDU in current embedding index: False


In [7]:
from src.rst_relation_selector import RSTRelationSelector

available_relations = {
    edge['relation']
    for edges in retriever.rst_adjacency.values()
    for edge in edges
}

selector = RSTRelationSelector(
    available_relations=available_relations,
)

selected = selector.select(
    valid_dataset[10]['query']
)

print(selected)

['Comparison', 'Contrast']


In [8]:
selected = selector.select(
    valid_dataset[10]['query']
)

print(selected)

['Comparison', 'Contrast']


In [9]:
from src.rgrag_annotation_retriever import RGRAGAnnotationRetriever

annotation_retriever = RGRAGAnnotationRetriever(
    workspace_path=project_root / 'graphrag_workspace',
    rst_edges_path=project_root / 'artifacts' / 'rgrag' / 'rst_relationship_edges.jsonl',
    consolidated_path=project_root / 'artifacts' / 'rgrag' / 'relationship_edu_consolidated.jsonl',
    community_level=2,
)

In [10]:
item = valid_dataset[10]

result = annotation_retriever.retrieve(
    item['query'],
    top_k=10,
)

print('Selected relations:', result['selected_relations'])
print('GraphRAG candidates:', result['graphrag_candidates'])
print('Annotation candidates:', result['annotation_candidates'])
print('Annotation new candidates:', result['annotation_new_candidates'])
print('RGRAG candidates:', result['rgrag_candidates'])
print('Annotation relationships:', result['annotation_relationships'])
print('Annotation edges:', result['annotation_edges'])

Selected relations: ['Comparison', 'Contrast']
GraphRAG candidates: 55
Annotation candidates: 6
Annotation new candidates: 0
RGRAG candidates: 55
Annotation relationships: 5
Annotation edges: 6


In [11]:
print()

for rank, retrieved in enumerate(result['ranked'], start=1):
    gold = any(
        normalize_text(evidence['fact'])
        in normalize_text(retrieved['text'])
        for evidence in item['evidence_list']
    )

    print(
        rank,
        'GOLD' if gold else '',
        retrieved['origin'],
        round(retrieved['score'], 4),
    )


1  graphrag 0.5446
2  graphrag 0.5424
3  graphrag 0.5237
4  graphrag 0.5227
5  graphrag 0.5163
6  graphrag 0.5148
7  graphrag 0.5129
8  graphrag 0.5017
9  graphrag 0.4993
10  graphrag 0.4941


In [12]:
query = valid_dataset[10]['query']
item = valid_dataset[10]

graphrag = annotation_retriever.rgrag.graphrag.retrieve_candidates(query)

structure = annotation_retriever.rgrag.get_rst_candidates(
    graphrag['relationship_ids']
)

graphrag_ids = {
    str(text_unit_id)
    for text_unit_id in graphrag['text_unit_ids']
}

text_units = {
    str(text_unit.id): text_unit
    for text_unit in annotation_retriever.rgrag.graphrag.text_units
}

In [14]:
matches = []

for edge in structure['edges']:
    key = (
        str(edge['relationship_id']),
        str(edge['document_id']),
    )

    context = annotation_retriever.rgrag.relationship_context.get(key)

    if context is None:
        continue

    for text_unit_id in context.get('text_unit_ids', []):
        text_unit_id = str(text_unit_id)

        if text_unit_id in graphrag_ids:
            continue

        text_unit = text_units.get(text_unit_id)

        if text_unit is None:
            continue

        for evidence in item['evidence_list']:
            if normalize_text(evidence['fact']) in normalize_text(text_unit.text):
                matches.append({
                    'relation': edge['relation'],
                    'nuclearity': edge['nuclearity'],
                    'seed_relationship_id': edge['seed_relationship_id'],
                    'relationship_id': edge['relationship_id'],
                    'text_unit_id': text_unit_id,
                    'fact': evidence['fact'],
                })

In [15]:
for match in matches:
    print('RELATION:', match['relation'])
    print('NUCLEARITY:', match['nuclearity'])
    print('FACT:', match['fact'])
    print('SEED:', match['seed_relationship_id'])
    print('TARGET:', match['relationship_id'])
    print()

RELATION: Background
NUCLEARITY: NS
FACT: “Hopefully we can go one better this year,” he added, recalling the 2-0 defeat by Canada in 2022.
SEED: b4c0cc6e-b7d2-4d4a-ab17-202ea4fae691
TARGET: 8cde3e88-adbb-4122-9119-eb974f93c76a

RELATION: Elaboration
NUCLEARITY: NS
FACT: “Hopefully we can go one better this year,” he added, recalling the 2-0 defeat by Canada in 2022.
SEED: b4c0cc6e-b7d2-4d4a-ab17-202ea4fae691
TARGET: 8cde3e88-adbb-4122-9119-eb974f93c76a

RELATION: Elaboration
NUCLEARITY: NS
FACT: “Hopefully we can go one better this year,” he added, recalling the 2-0 defeat by Canada in 2022.
SEED: b4c0cc6e-b7d2-4d4a-ab17-202ea4fae691
TARGET: 8cde3e88-adbb-4122-9119-eb974f93c76a

RELATION: Joint
NUCLEARITY: NN
FACT: “Hopefully we can go one better this year,” he added, recalling the 2-0 defeat by Canada in 2022.
SEED: 57479d93-5ab6-42e9-b5c3-80f134a13215
TARGET: 8cde3e88-adbb-4122-9119-eb974f93c76a

RELATION: Temporal
NUCLEARITY: NN
FACT: “Hopefully we can go one better this year,” he 

In [16]:
for match in matches:
    for edge in structure['edges']:
        if (
            edge['seed_relationship_id'] == match['seed_relationship_id']
            and edge['relationship_id'] == match['relationship_id']
            and edge['relation'] == match['relation']
        ):
            print('RELATION:', edge['relation'])
            print('NUCLEARITY:', edge['nuclearity'])
            print('SOURCE ROLE:', edge.get('source_role'))
            print('TARGET ROLE:', edge.get('target_role'))
            print('FACT:', match['fact'])
            print()
            break

RELATION: Background
NUCLEARITY: NS
SOURCE ROLE: Nucleus
TARGET ROLE: Satellite
FACT: “Hopefully we can go one better this year,” he added, recalling the 2-0 defeat by Canada in 2022.

RELATION: Elaboration
NUCLEARITY: NS
SOURCE ROLE: Satellite
TARGET ROLE: Nucleus
FACT: “Hopefully we can go one better this year,” he added, recalling the 2-0 defeat by Canada in 2022.

RELATION: Elaboration
NUCLEARITY: NS
SOURCE ROLE: Satellite
TARGET ROLE: Nucleus
FACT: “Hopefully we can go one better this year,” he added, recalling the 2-0 defeat by Canada in 2022.

RELATION: Joint
NUCLEARITY: NN
SOURCE ROLE: Nucleus
TARGET ROLE: Nucleus
FACT: “Hopefully we can go one better this year,” he added, recalling the 2-0 defeat by Canada in 2022.

RELATION: Temporal
NUCLEARITY: NN
SOURCE ROLE: Nucleus
TARGET ROLE: Nucleus
FACT: “Hopefully we can go one better this year,” he added, recalling the 2-0 defeat by Canada in 2022.

RELATION: Joint
NUCLEARITY: NN
SOURCE ROLE: Nucleus
TARGET ROLE: Nucleus
FACT: “Hope

In [17]:
import json

results_path = project_root / 'artifacts' / 'evaluation' / 'rgrag_structure_results.jsonl'

with open(results_path, 'r', encoding='utf-8') as file:
    first_result = json.loads(next(line for line in file if line.strip()))

print('FIELDS')
print()

for key in first_result:
    print(key)

FIELDS

dataset_index
query
question_type
gold_count
graphrag_candidates
rst_candidates
rst_new_candidates
rgrag_candidates
graphrag_found
rgrag_found
graphrag_candidate_recall
rgrag_candidate_recall
graphrag_candidate_complete
rgrag_candidate_complete
new_gold_found
graphrag_metrics
rgrag_metrics
evidence
graphrag_top10
rgrag_top10


In [18]:
improved_result = None

with open(results_path, 'r', encoding='utf-8') as file:
    for line in file:
        if not line.strip():
            continue

        result = json.loads(line)

        if result['new_gold_found'] > 0:
            improved_result = result
            break

print('DATASET INDEX:', improved_result['dataset_index'])
print('QUERY:', improved_result['query'])
print()
print('EVIDENCE')
print()

for item in improved_result['evidence']:
    print(item)
    print()

DATASET INDEX: 8
QUERY: What is the name of the organization discussed in TechCrunch articles that, despite its financial instability, is recognized for creating ChatGPT, which is both a priority and a platform for ongoing innovations, and is planning to enhance its capabilities with the release of GPT-4 and associated APIs?

EVIDENCE

{'source': 'TechCrunch', 'title': 'WTF is going on at OpenAI? We have theories', 'fact': 'Despite being the hottest tech company in the world right now and everyone talking about ChatGPT, OpenAI isn’t exactly a sound business.', 'graphrag_candidate_found': False, 'rgrag_candidate_found': True, 'graphrag_rank': None, 'rgrag_rank': 37, 'rgrag_origin': 'rst'}

{'source': 'TechCrunch', 'title': 'How the OpenAI fiasco could bolster Meta and the ‘open AI’ movement', 'fact': 'It has been a whirlwind four days for OpenAI, the generative AI poster child behind the smash hit ChatGPT.', 'graphrag_candidate_found': True, 'rgrag_candidate_found': True, 'graphrag_rank

In [19]:
from collections import Counter, defaultdict

positive_edges = Counter()
all_edges = Counter()

positive_gold_by_relation = defaultdict(set)
positive_queries_by_relation = defaultdict(set)

improved_results = []

with open(results_path, 'r', encoding='utf-8') as file:
    for line in file:
        if not line.strip():
            continue

        result = json.loads(line)

        if result['new_gold_found'] > 0:
            improved_results.append(result)

print('Improved queries:', len(improved_results))

Improved queries: 181


In [20]:
text_units = {
    str(text_unit.id): text_unit
    for text_unit in annotation_retriever.rgrag.graphrag.text_units
}

for index, result in enumerate(improved_results, start=1):
    query = result['query']
    dataset_index = result['dataset_index']

    new_gold = [
        evidence
        for evidence in result['evidence']
        if not evidence['graphrag_candidate_found']
        and evidence['rgrag_candidate_found']
    ]

    graphrag = annotation_retriever.rgrag.graphrag.retrieve_candidates(query)

    structure = annotation_retriever.rgrag.get_rst_candidates(
        graphrag['relationship_ids']
    )

    graphrag_ids = {
        str(text_unit_id)
        for text_unit_id in graphrag['text_unit_ids']
    }

    for edge in structure['edges']:
        annotation = (
            edge['relation'],
            edge['nuclearity'],
            edge.get('source_role'),
            edge.get('target_role'),
        )

        all_edges[annotation] += 1

        key = (
            str(edge['relationship_id']),
            str(edge['document_id']),
        )

        context = annotation_retriever.rgrag.relationship_context.get(key)

        if context is None:
            continue

        matched_facts = set()

        for text_unit_id in context.get('text_unit_ids', []):
            text_unit_id = str(text_unit_id)

            if text_unit_id in graphrag_ids:
                continue

            text_unit = text_units.get(text_unit_id)

            if text_unit is None:
                continue

            normalized_text = normalize_text(text_unit.text)

            for evidence in new_gold:
                fact = evidence['fact']

                if normalize_text(fact) in normalized_text:
                    matched_facts.add(fact)

        if not matched_facts:
            continue

        positive_edges[annotation] += 1
        positive_queries_by_relation[annotation].add(dataset_index)

        for fact in matched_facts:
            positive_gold_by_relation[annotation].add(
                (dataset_index, fact)
            )

    if index % 25 == 0:
        print(f'Processed: {index}/{len(improved_results)}')

Processed: 25/181
Processed: 50/181
Processed: 75/181
Processed: 100/181
Processed: 125/181
Processed: 150/181
Processed: 175/181


In [21]:
print()
print('ANNOTATION ANALYSIS')
print()

for annotation, positive_count in positive_edges.most_common():
    total_count = all_edges[annotation]

    gold_count = len(
        positive_gold_by_relation[annotation]
    )

    query_count = len(
        positive_queries_by_relation[annotation]
    )

    positive_rate = positive_count / total_count

    relation, nuclearity, source_role, target_role = annotation

    print(
        f'{relation} | {nuclearity} | '
        f'{source_role} -> {target_role}'
    )
    print(f'  Positive edges: {positive_count}')
    print(f'  All edges: {total_count}')
    print(f'  Positive rate: {positive_rate:.4f}')
    print(f'  Unique gold facts: {gold_count}')
    print(f'  Queries: {query_count}')
    print()


ANNOTATION ANALYSIS

Joint | NN | Nucleus -> Nucleus
  Positive edges: 3344
  All edges: 405899
  Positive rate: 0.0082
  Unique gold facts: 175
  Queries: 153

Temporal | NN | Nucleus -> Nucleus
  Positive edges: 622
  All edges: 26383
  Positive rate: 0.0236
  Unique gold facts: 48
  Queries: 38

Topic-Change | NN | Nucleus -> Nucleus
  Positive edges: 474
  All edges: 7902
  Positive rate: 0.0600
  Unique gold facts: 45
  Queries: 40

Elaboration | NS | Nucleus -> Satellite
  Positive edges: 330
  All edges: 20885
  Positive rate: 0.0158
  Unique gold facts: 32
  Queries: 32

Background | SN | Satellite -> Nucleus
  Positive edges: 261
  All edges: 9600
  Positive rate: 0.0272
  Unique gold facts: 29
  Queries: 29

Background | SN | Nucleus -> Satellite
  Positive edges: 154
  All edges: 6003
  Positive rate: 0.0257
  Unique gold facts: 12
  Queries: 12

Elaboration | NS | Satellite -> Nucleus
  Positive edges: 144
  All edges: 17306
  Positive rate: 0.0083
  Unique gold facts: 54


In [22]:
dense_all_ids = {
    str(text_unit.id)
    for text_unit in annotation_retriever.rgrag.graphrag.text_units
}

print('Dense-All candidates:', len(dense_all_ids))

Dense-All candidates: 1533


In [23]:
item = valid_dataset[10]
query = item['query']

dense_ranked = annotation_retriever.rgrag.ranker.rank(
    query=query,
    text_unit_ids=dense_all_ids,
    baseline_ids=dense_all_ids,
    rst_ids=set(),
    top_k=10,
)

In [24]:
for rank, retrieved in enumerate(dense_ranked, start=1):
    gold = any(
        normalize_text(evidence['fact'])
        in normalize_text(retrieved['text'])
        for evidence in item['evidence_list']
    )

    print(
        rank,
        'GOLD' if gold else '',
        round(retrieved['score'], 4),
    )

1  0.5446
2  0.5424
3  0.5237
4  0.5227
5  0.5163
6  0.5162
7  0.5148
8  0.5129
9  0.5024
10  0.5017


In [25]:
dense_ranked_all = annotation_retriever.rgrag.ranker.rank(
    query=query,
    text_unit_ids=dense_all_ids,
    baseline_ids=dense_all_ids,
    rst_ids=set(),
    top_k=len(dense_all_ids),
)

print()

for evidence in item['evidence_list']:
    rank = None

    for index, retrieved in enumerate(dense_ranked_all, start=1):
        if normalize_text(evidence['fact']) in normalize_text(retrieved['text']):
            rank = index
            break

    print('SOURCE:', evidence['source'])
    print('FACT:', evidence['fact'])
    print('DENSE RANK:', rank)
    print()


SOURCE: The Age
FACT: “Hopefully we can go one better this year,” he added, recalling the 2-0 defeat by Canada in 2022.
DENSE RANK: 28

SOURCE: Sporting News
FACT: England will no doubt be looking to echo the spirit of 2019, when they beat the All Blacks 19-7 to reach the final, but in South Africa, they face a side who have taken their game to new heights just to reach the semis.
DENSE RANK: 14

